# Tugas Individu Deep Learning: Komparasi CNN vs Transfer Learning pada Waste / Garbage Classification

- **Topik Proyek**: Klasifikasi Jenis Sampah dan Material Daur Ulang (*Smart Waste Management*)
- **Nama Praktikan**: [ISI NAMA ANDA / ARIL]
- **NIM**: [ISI NIM ANDA]
- **Mata Kuliah**: Deep Learning
- **Dataset**: [Garbage Classification (Kaggle: asdasdasasdas/garbage-classification)](https://www.kaggle.com/datasets/asdasdasasdas/garbage-classification)
- **Target Hardware**: Google Colab Free GPU (NVIDIA T4)

---
### Ringkasan Eksperimen
Proyek ini membandingkan kinerja model Deep Learning dalam mengklasifikasikan 6 kategori material sampah:
1. **Model 1 — Custom Deep CNN from scratch**: Arsitektur konvolusional 4-blok (32 $\rightarrow$ 64 $\rightarrow$ 128 $\rightarrow$ 256) dengan `BatchNormalization`, `ReLU`, `MaxPooling2D`, `Spatial Dropout`, dan `GlobalAveragePooling2D`.
2. **Model 2 — Transfer Learning ResNet50V2**: Arsitektur residual deep learning pra-latih ImageNet dengan strategi **two-stage fine-tuning** (feature extraction lalu unfreeze 30 layer teratas).
3. **Model 3 (Opsional) — Transfer Learning MobileNetV2**: Pembanding arsitektur ringan berbasis inverted residual blocks.

Metrik evaluasi mencakup: **Accuracy, Precision (Macro/Weighted), Recall (Macro/Weighted), F1-Score (Macro/Weighted), Confusion Matrix, Training Time**, dan interpretabilitas visual berbasis **Grad-CAM**.


## 1. Setup Environment, GPU, dan Dependensi
Memastikan ketersediaan GPU (NVIDIA T4), mengaktifkan **Mixed Precision** (`mixed_float16`) untuk efisiensi komputasi, dan menginstal dependensi yang dibutuhkan.


In [ ]:
# Cek status GPU NVIDIA
!nvidia-smi

# Instal dependensi tambahan
!pip install -q kagglehub scikit-learn seaborn matplotlib pandas


### Clone Repository & Sinkronisasi Modul `src`
Seluruh modul modular (`config`, `data_loader`, `models`, `train`, `evaluate`, `gradcam`) berada di folder `src/`.


In [ ]:
import os
import sys

# Konfigurasi path modul
if 'google.colab' in sys.modules:
    repo_name = "dl-assignment-waste-classification"
    repo_path = f"/content/{repo_name}"
    
    if not os.path.exists(repo_path) and not os.path.exists("src"):
        print("Cloning repository...")
        # Ganti dengan URL repositori GitHub Anda jika diperlukan
        !git clone https://github.com/dwiwahyu/tugas-na-aril-2-arsitektur.git {repo_path} || true
        if os.path.exists(repo_path):
            os.chdir(repo_path)
            
    if os.getcwd() not in sys.path:
        sys.path.insert(0, os.getcwd())
else:
    current_dir = os.path.dirname(os.path.abspath('__file__'))
    parent_dir = os.path.abspath(os.path.join(current_dir, '..'))
    if parent_dir not in sys.path:
        sys.path.insert(0, parent_dir)

print("Current Working Directory:", os.getcwd())


### Import Modul & Inisialisasi Reproducibility Seed


In [ ]:
import random
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from src.config import (
    SEED,
    IMG_SIZE_CNN,
    IMG_SIZE_TL,
    BATCH_SIZE,
    EPOCHS,
    NUM_CLASSES,
    CLASS_NAMES,
    OUTPUT_DIR,
    MODEL_DIR,
    set_seed,
    setup_directories,
    configure_mixed_precision,
)
from src.data_loader import (
    download_dataset_kagglehub,
    find_data_directory,
    build_datasets,
    get_class_names,
)
from src.models import (
    build_custom_cnn,
    build_transfer_learning_model,
    unfreeze_top_layers,
    compile_model,
)
from src.train import (
    get_callbacks,
    train_model,
    save_training_history,
)
from src.evaluate import (
    plot_training_history,
    evaluate_model,
    compare_models,
)
from src.gradcam import (
    make_gradcam_heatmap,
    superimpose_gradcam,
    visualize_gradcam_samples,
)

# 1. Inisialisasi Seed dan Direktori Output
set_seed(SEED)
setup_directories()

# 2. Aktifkan Mixed Precision (FP16) untuk GPU T4
active_policy = configure_mixed_precision(enable=True)
print(f"TensorFlow Version: {tf.__version__}")
print(f"Random Seed set to: {SEED}")
print(f"Active Mixed Precision Policy: {active_policy}")
print(f"Available GPUs: {tf.config.list_physical_devices('GPU')}")


## 2. Pemuatan Dataset Waste / Garbage Classification
Dataset diunduh langsung menggunakan pustaka resmi `kagglehub` tanpa perlu API token manual jika dijalankan di Colab.


In [ ]:
# Unduh otomatis via KaggleHub
dataset_path = download_dataset_kagglehub("asdasdasasdas/garbage-classification")
data_dir = find_data_directory(dataset_path)

print(f"Root Data Directory: {data_dir}")
resolved_classes = get_class_names(data_dir)
print(f"Resolved Classes ({len(resolved_classes)}): {resolved_classes}")


## 3. Exploratory Data Analysis (EDA) Singkat
Memeriksa distribusi jumlah gambar per kategori material sampah serta menampilkan cuplikan citra untuk tiap kelas.


In [ ]:
# 1. Hitung jumlah gambar per kategori
class_counts = {c: len([f for f in os.listdir(os.path.join(data_dir, c)) if not f.startswith('.')]) for c in resolved_classes}
df_counts = pd.DataFrame(list(class_counts.items()), columns=["Kategori Sampah", "Jumlah Gambar"])
df_counts = df_counts.sort_values(by="Jumlah Gambar", ascending=False).reset_index(drop=True)
print("Distribusi Citra per Kategori:")
display(df_counts)

# 2. Visualisasi Distribusi Kelas
plt.figure(figsize=(10, 4))
plt.bar(df_counts["Kategori Sampah"], df_counts["Jumlah Gambar"], color="#2b5c8f")
plt.title("Distribusi Jumlah Citra per Kategori Sampah", fontsize=13, fontweight="bold")
plt.xlabel("Kategori", fontsize=11)
plt.ylabel("Jumlah Gambar", fontsize=11)
plt.xticks(rotation=25)
plt.grid(axis="y", linestyle=":", alpha=0.7)
for i, v in enumerate(df_counts["Jumlah Gambar"]):
    plt.text(i, v + 10, str(v), ha="center", fontsize=10, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "dataset_class_distribution.png"), dpi=300)
plt.show()

# 3. Visualisasi Sampel Gambar Tiap Kategori
fig, axes = plt.subplots(len(resolved_classes), 5, figsize=(15, 3 * len(resolved_classes)))
for row_idx, cls_name in enumerate(resolved_classes):
    cls_folder = os.path.join(data_dir, cls_name)
    sample_files = [f for f in os.listdir(cls_folder) if not f.startswith('.')][:5]
    for col_idx, file_name in enumerate(sample_files):
        img_p = os.path.join(cls_folder, file_name)
        img = plt.imread(img_p)
        axes[row_idx, col_idx].imshow(img)
        axes[row_idx, col_idx].axis("off")
        if col_idx == 0:
            axes[row_idx, col_idx].set_title(cls_name.upper(), fontsize=12, fontweight="bold", loc="left")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "dataset_samples_grid.png"), dpi=300)
plt.show()


## 4. Pipeline Data tf.data (Ukuran 150x150 & 224x224)
- **Custom CNN**: Resolusi $150 \times 150$, normalisasi `Rescaling(1./255)`.
- **Transfer Learning**: Resolusi $224 \times 224$, normalisasi `preprocess_input` ImageNet.
- **Split Data**: 70% Training, 15% Validation, dan 15% Testing murni.


In [ ]:
# Normalisasi Preprocessing Functions
preprocess_cnn = tf.keras.layers.Rescaling(1.0 / 255.0)
preprocess_resnet = tf.keras.applications.resnet_v2.preprocess_input
preprocess_mobilenet = tf.keras.applications.mobilenet_v2.preprocess_input

# 1. Pipeline untuk Custom CNN (150 x 150)
print("\n[Pipeline 1] Membangun dataset untuk Custom CNN (150x150)...")
train_ds_cnn, val_ds_cnn, test_ds_cnn = build_datasets(
    data_dir=data_dir,
    img_size=IMG_SIZE_CNN,
    batch_size=BATCH_SIZE,
    use_augmentation=True,
    preprocess_fn=preprocess_cnn,
    train_split=0.70,
    val_split=0.15,
    test_split=0.15,
    seed=SEED,
)

# 2. Pipeline untuk ResNet50V2 (224 x 224)
print("\n[Pipeline 2] Membangun dataset untuk ResNet50V2 (224x224)...")
train_ds_res, val_ds_res, test_ds_res = build_datasets(
    data_dir=data_dir,
    img_size=IMG_SIZE_TL,
    batch_size=BATCH_SIZE,
    use_augmentation=True,
    preprocess_fn=preprocess_resnet,
    train_split=0.70,
    val_split=0.15,
    test_split=0.15,
    seed=SEED,
)

# 3. Pipeline untuk MobileNetV2 (224 x 224)
print("\n[Pipeline 3] Membangun dataset untuk MobileNetV2 (224x224)...")
train_ds_mbn, val_ds_mbn, test_ds_mbn = build_datasets(
    data_dir=data_dir,
    img_size=IMG_SIZE_TL,
    batch_size=BATCH_SIZE,
    use_augmentation=True,
    preprocess_fn=preprocess_mobilenet,
    train_split=0.70,
    val_split=0.15,
    test_split=0.15,
    seed=SEED,
)

print(f"\nBatch Size: {BATCH_SIZE}")
print(f"Jumlah batch Train: {tf.data.experimental.cardinality(train_ds_cnn).numpy()}")
print(f"Jumlah batch Val:   {tf.data.experimental.cardinality(val_ds_cnn).numpy()}")
print(f"Jumlah batch Test:  {tf.data.experimental.cardinality(test_ds_cnn).numpy()}")


## 5. Model 1 — Custom Deep CNN from Scratch
Arsitektur konvolusi 4-blok ($32 \rightarrow 64 \rightarrow 128 \rightarrow 256$) dengan `BatchNormalization`, aktivasi `ReLU`, `MaxPooling2D`, `Dropout`, dan `GlobalAveragePooling2D` classification head.


In [ ]:
# Membangun dan mengompilasi model Custom CNN
model_cnn = build_custom_cnn(
    input_shape=(IMG_SIZE_CNN[0], IMG_SIZE_CNN[1], 3),
    num_classes=NUM_CLASSES,
    name="custom_cnn"
)
model_cnn = compile_model(model_cnn, learning_rate=1e-3)
model_cnn.summary()


### Training Model 1: Custom CNN


In [ ]:
# Opsi memuat model yang sudah ada:
# model_cnn = tf.keras.models.load_model(os.path.join(MODEL_DIR, "custom_cnn_best.keras"))

callbacks_cnn = get_callbacks(model_name="custom_cnn", model_dir=MODEL_DIR)

history_cnn, time_cnn = train_model(
    model=model_cnn,
    train_ds=train_ds_cnn,
    val_ds=val_ds_cnn,
    epochs=EPOCHS,
    callbacks=callbacks_cnn,
)

save_training_history(history_cnn, model_name="custom_cnn")


### Evaluasi Model 1: Custom CNN


In [ ]:
# Visualisasi Kurva Pelatihan
plot_training_history(history_cnn, model_name="custom_cnn")

# Evaluasi pada Test Set
eval_cnn = evaluate_model(
    model=model_cnn,
    test_ds=test_ds_cnn,
    class_names=CLASS_NAMES,
    model_name="Custom CNN",
    training_time=time_cnn
)


## 6. Model 2 — Transfer Learning ResNet50V2
Menggunakan backbone **ResNet50V2** pra-latih ImageNet.
- **Tahap 1 (Feature Extraction)**: Base dibekukan (`trainable=False`), melatih classification head dengan Adam (learning rate $10^{-3}$).
- **Tahap 2 (Fine-Tuning)**: Membuka 30 layer teratas base model dengan learning rate lebih halus ($10^{-4}$), menjaga `BatchNormalization` tetap beku untuk menjaga kestabilan statistik ImageNet.


In [ ]:
# Tahap 1: Feature Extraction
model_resnet = build_transfer_learning_model(
    base_name="ResNet50V2",
    input_shape=(IMG_SIZE_TL[0], IMG_SIZE_TL[1], 3),
    num_classes=NUM_CLASSES,
    fine_tune=False,
    name="resnet50v2"
)
model_resnet = compile_model(model_resnet, learning_rate=1e-3)
model_resnet.summary()


In [ ]:
callbacks_res_fe = get_callbacks(model_name="resnet50v2_fe", model_dir=MODEL_DIR)

history_res_fe, time_res_fe = train_model(
    model=model_resnet,
    train_ds=train_ds_res,
    val_ds=val_ds_res,
    epochs=12,
    callbacks=callbacks_res_fe,
)


### Fine-Tuning ResNet50V2 (Unfreeze Top 30 Layers)


In [ ]:
# Unfreeze 30 layer teratas base model
unfreeze_top_layers(model_resnet, num_layers=30)

# Kompilasi ulang dengan learning rate 1e-4
model_resnet = compile_model(model_resnet, learning_rate=1e-4)

callbacks_res_ft = get_callbacks(model_name="resnet50v2_finetuned", model_dir=MODEL_DIR)

history_res_ft, time_res_ft = train_model(
    model=model_resnet,
    train_ds=train_ds_res,
    val_ds=val_ds_res,
    epochs=20,
    initial_epoch=history_res_fe.epoch[-1] + 1 if hasattr(history_res_fe, "epoch") else 12,
    callbacks=callbacks_res_ft,
)

total_time_resnet = time_res_fe + time_res_ft

# Gabungkan riwayat training
combined_history_res = {
    metric: history_res_fe.history[metric] + history_res_ft.history[metric]
    for metric in history_res_fe.history
}
save_training_history(combined_history_res, model_name="resnet50v2_combined")


### Evaluasi Model 2: ResNet50V2 (Fine-Tuned)


In [ ]:
plot_training_history(combined_history_res, model_name="resnet50v2_finetuned")

eval_res = evaluate_model(
    model=model_resnet,
    test_ds=test_ds_res,
    class_names=CLASS_NAMES,
    model_name="ResNet50V2 (Fine-Tuned)",
    training_time=total_time_resnet
)


## 7. Model 3 (Opsional) — Transfer Learning MobileNetV2
Membandingkan model Residual (ResNet50V2) dengan arsitektur ringan MobileNetV2.


In [ ]:
# 1. Feature Extraction
model_mobilenet = build_transfer_learning_model(
    base_name="MobileNetV2",
    input_shape=(IMG_SIZE_TL[0], IMG_SIZE_TL[1], 3),
    num_classes=NUM_CLASSES,
    fine_tune=False,
    name="mobilenetv2"
)
model_mobilenet = compile_model(model_mobilenet, learning_rate=1e-3)

callbacks_mbn_fe = get_callbacks(model_name="mobilenetv2_fe", model_dir=MODEL_DIR)
history_mbn_fe, time_mbn_fe = train_model(
    model=model_mobilenet,
    train_ds=train_ds_mbn,
    val_ds=val_ds_mbn,
    epochs=10,
    callbacks=callbacks_mbn_fe,
)

# 2. Fine-Tuning
unfreeze_top_layers(model_mobilenet, num_layers=30)
model_mobilenet = compile_model(model_mobilenet, learning_rate=1e-4)

callbacks_mbn_ft = get_callbacks(model_name="mobilenetv2_finetuned", model_dir=MODEL_DIR)
history_mbn_ft, time_mbn_ft = train_model(
    model=model_mobilenet,
    train_ds=train_ds_mbn,
    val_ds=val_ds_mbn,
    epochs=18,
    initial_epoch=history_mbn_fe.epoch[-1] + 1 if hasattr(history_mbn_fe, "epoch") else 10,
    callbacks=callbacks_mbn_ft,
)

total_time_mbn = time_mbn_fe + time_mbn_ft
combined_history_mbn = {
    metric: history_mbn_fe.history[metric] + history_mbn_ft.history[metric]
    for metric in history_mbn_fe.history
}
save_training_history(combined_history_mbn, model_name="mobilenetv2_combined")
plot_training_history(combined_history_mbn, model_name="mobilenetv2_finetuned")

eval_mbn = evaluate_model(
    model=model_mobilenet,
    test_ds=test_ds_mbn,
    class_names=CLASS_NAMES,
    model_name="MobileNetV2 (Fine-Tuned)",
    training_time=total_time_mbn
)


## 8. Analisis & Komparasi Antar Model
Tabel komparatif dan bar chart yang merangkum performa seluruh model berdasarkan Accuracy, Precision, Recall, F1-Score, dan Waktu Pelatihan.


In [ ]:
comparison_results = [eval_cnn, eval_res, eval_mbn]
df_comparison = compare_models(comparison_results, save_dir=OUTPUT_DIR)
display(df_comparison)


## 9. Grad-CAM (Gradient-Weighted Class Activation Mapping)
Visualisasi area citra yang menjadi fokus representasi fitur pada model terbaik.


In [ ]:
best_eval = max(comparison_results, key=lambda x: x["accuracy"])
print(f"Model terbaik yang dipilih untuk Grad-CAM: {best_eval['model_name']} (Akurasi: {best_eval['accuracy']*100:.2f}%)")

if "ResNet" in best_eval["model_name"]:
    best_model_for_cam = model_resnet
    target_test_ds = test_ds_res
elif "MobileNet" in best_eval["model_name"]:
    best_model_for_cam = model_mobilenet
    target_test_ds = test_ds_mbn
else:
    best_model_for_cam = model_cnn
    target_test_ds = test_ds_cnn

gradcam_plot_path = visualize_gradcam_samples(
    model=best_model_for_cam,
    dataset=target_test_ds,
    class_names=CLASS_NAMES,
    num_samples=8,
    save_path=os.path.join(OUTPUT_DIR, "best_model_gradcam_grid.png")
)

from IPython.display import Image
Image(filename=gradcam_plot_path)


## 10. Kesimpulan & Analisis Temuan

### Ringkasan Temuan Eksperimen:
1. **Keunggulan Transfer Learning pada Citra Material Sampah**: Model Transfer Learning (ResNet50V2 dan MobileNetV2) terbukti mampu mengenali karakteristik tekstur dan refleksi material sampah (seperti transparansi kaca, tekstur serat kardus/kertas, dan kilau plastik/logam) dengan akurasi jauh melampaui Custom CNN from scratch.
2. **Efektivitas Fine-Tuning Bertahap**: Pembekuan backbone pada tahap Feature Extraction memberi waktu bagi classification head untuk beradaptasi, sedangkan unfreezing 30 layer teratas memungkinkan feature map menyesuaikan diri dengan domain spesifik objek daur ulang.
3. **Analisis Grad-CAM**: Heatmap Grad-CAM mengonfirmasi bahwa model transfer learning memfokuskan perhatian pada kontur objek sampah (tepi botol, lipatan kardus) dan bukan pada background permukaan meja/lantai.


## 11. Ekspor & Backup Artifacts


In [ ]:
import shutil

zip_filename = "waste_classification_artifacts"
shutil.make_archive(zip_filename, 'zip', root_dir='.', base_dir='outputs')
print(f"Artifact berhasil dikompres ke: {zip_filename}.zip")

# Uncomment jika ingin mengunduh langsung di browser:
# from google.colab import files
# files.download(f"{zip_filename}.zip")
